<a href="https://colab.research.google.com/github/Irfan-code-cloud/ML-Internship-flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### ML Task Type: Scoring & Prioritized Ranking

To optimize our content-refresh workflow, we frame this problem as a **Scoring (Regression)** task that feeds directly into a **Ranking** system:

1. **The Scoring Task (Regression):**
   Our model will predict a continuous value the **Expected CTR** for a given page based on its position, query intent, and content metadata. By subtracting the page's actual, observed CTR from this predicted baseline, we calculate a custom metric: the **CTR Gap (Opportunity Score)**.

   <br>
   $$\text{Opportunity Score} = \text{Predicted Expected CTR} - \text{Actual Observed CTR}$$
   <br>

2. **The Ranking Task:**
   Once every page has an Opportunity Score, we sort the pages in descending order. This produces a prioritized queue.

* **Why this beats alternative ML framings:**
  * **Instead of Classification:** Binary classification ("Needs Update" vs. "Keep") lacks granularity. Scoring allows us to quantify *how much* a page is underperforming, preventing editorial teams from wasting time on pages with negligible room for improvement.
  * **Instead of Clustering:** Unsupervised clustering might group similar pages together, but it cannot rank them by potential traffic recovery or measure execution priority.

In [1]:
# =====================================================================
# SYSTEM CONFIGURATION: TASK TYPE DEFINITION
# =====================================================================

pipeline_config = {
    "primary_ml_task": "Scoring (Regression)",
    "downstream_application": "Prioritized Ranking Queue",
    "target_variable": "ctr_gap",
    "mathematical_definition": "expected_ctr - observed_ctr",
    "output_action": "Generate a sorted backlog of pages for copywriting optimization"
}

# Print configuration diagnostics to verify execution
print("✅ ML Task Type successfully registered in notebook session.")
print(f"🔹 Task Type : {pipeline_config['primary_ml_task']}")
print(f"🔹 Downstream: {pipeline_config['downstream_application']}")
print(f"🔹 Target Col: {pipeline_config['target_variable']}")

✅ ML Task Type successfully registered in notebook session.
🔹 Task Type : Scoring (Regression)
🔹 Downstream: Prioritized Ranking Queue
🔹 Target Col: ctr_gap


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target and Proxy Formulation

To model this, we define a clear relationship between our predicted target and our business proxy:

1. **What we predict (The Machine Learning Target):**
   Our model will predict **Expected CTR** ($\hat{y}$). This is a continuous numerical target trained directly on the historical, observed `ctr` column in our dataset.
   
2. **Where the label comes from (Observed Outcome):**
   Our training labels ($y$) are the actual, historical `ctr` values recorded in our dataset. The model learns what a "normal" CTR looks like by training on real search behavior mapped against positional and page characteristics (e.g., page intent, word count, and position).

3. **Our Business Proxy (The Defined Rule):**
   The final prioritized metric is the **CTR Gap (Opportunity Score)**, which is calculated downstream using a post-prediction rule:
   
   $$\text{CTR Gap} = \text{Model Predicted Expected CTR} (\hat{y}) - \text{Actual Observed CTR} (y)$$

* **Why we use this proxy framework:**
  Using the raw `ctr` alone doesn't flag underperformance. A $0.5\%$ CTR is outstanding for position 15, but terrible for position 2. By training the model to predict what a page *should* be getting based on its position and content characteristics, we can subtract its actual performance to isolate true, high-yield optimization targets.

In [6]:
# =====================================================================
# SYSTEM CONFIGURATION: TARGET AND PROXY SCHEMAS
# =====================================================================

target_schema = {
    "model_prediction_target": {
        "column_name": "ctr",
        "type": "observed_outcome",
        "data_type": "float64",
        "description": "The actual Click-Through Rate recorded in Google Search Console."
    },
    "business_proxy_target": {
        "column_name": "ctr_gap",
        "type": "derived_rule",
        "data_type": "float64",
        "description": "The calculated difference between the model's predicted CTR and the actual observed CTR."
    }
}

print("--- 🎯 Target & Proxy Verification ---")
print(f"🔹 ML Target Variable   : {target_schema['model_prediction_target']['column_name']} ({target_schema['model_prediction_target']['type']})")
print(f"🔹 Business Proxy Target: {target_schema['business_proxy_target']['column_name']} ({target_schema['business_proxy_target']['type']})")

--- 🎯 Target & Proxy Verification ---
🔹 ML Target Variable   : ctr (observed_outcome)
🔹 Business Proxy Target: ctr_gap (derived_rule)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success Metric: Mean Absolute Error (MAE)

For our primary scoring model (predicting **Expected CTR**), the single evaluation metric we will use and defend is **Mean Absolute Error (MAE)**.

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Where:
* $y_i$ is the actual, observed historical CTR.
* $\hat{y}_i$ is our model's predicted Expected CTR.

#### Why we choose and defend MAE:
1. **Business Interpretability:** MAE measures the average error in the exact same unit of measurement as our target variable (percentage points of CTR). Unlike Mean Squared Error (MSE), which penalizes outliers quadratically and outputs abstract squared units, MAE maps directly to real-world impact.
2. **Direct Trust Building:** If our model achieves an MAE of $0.40\%$, we can tell our copywriters: *"Our baseline predictions are, on average, within $\pm0.4\%$ of actual organic behavior."* This allows them to trust the priority queue.

#### What number means "good"?
* **Baseline Goal:** An MAE of **$< 0.80\%$** (0.8 percentage points).
* **Target "Good" Threshold:** An MAE of **$< 0.40\%$** (0.4 percentage points).
Given that a vast majority of organic listings sit between $0.1\%$ and $10\%$ CTR, keeping our average error under half a percentage point ensures that our derived **CTR Gap** is mathematically meaningful and not dominated by model noise.

In [7]:
# =====================================================================
# SYSTEM CONFIGURATION: SUCCESS METRIC CALCULATOR
# =====================================================================
from sklearn.metrics import mean_absolute_error

# Simulated actual CTRs from our dataset vs. what our model predicts
actual_ctrs    = [4.5, 1.2, 0.8, 12.0, 2.5]
predicted_ctrs = [4.1, 1.5, 0.9, 11.2, 2.1]

# Compute the Mean Absolute Error (MAE)
calculated_mae = mean_absolute_error(actual_ctrs, predicted_ctrs)
success_threshold = 0.50  # We want our predictions to be within 0.5% CTR of reality

model_is_acceptable = calculated_mae <= success_threshold

print("--- 📊 Evaluation Metric & Success Thresholds ---")
print(f"🔹 Metric Selected     : Mean Absolute Error (MAE)")
print(f"🔹 Simulated MAE Score : {calculated_mae:.3f}% (Average prediction error)")
print(f"🔹 Good Model Threshold: <= {success_threshold:.2f}%")
print(f"🔹 Performance Verdict : {'✅ PASSED' if model_is_acceptable else '❌ FAILED'}")

--- 📊 Evaluation Metric & Success Thresholds ---
🔹 Metric Selected     : Mean Absolute Error (MAE)
🔹 Simulated MAE Score : 0.400% (Average prediction error)
🔹 Good Model Threshold: <= 0.50%
🔹 Performance Verdict : ✅ PASSED


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis: The Unique URL (Page-Level)

In our project, the **unit of analysis is one unique anonymized page (URL)**.

#### Why this is our Unit of Analysis:
SEO content refreshes (such as rewriting titles, updating meta descriptions, or optimizing copy length) are executed page-by-page. Therefore, our model's features and target labels must exist at the individual URL level to be actionable.

A single row in our dataset represents one unique page, combining:
1. **Search Console Performance Metrics (Features & Labels):** `avg_position`, `impressions_90d`, and our observed target `ctr`.
2. **On-Page Metadata (Categorical Context):** `content_type`, `main_intent`, and `word_count`.

In [8]:
import os
import sys
import subprocess
import pandas as pd

# =====================================================================
# SYSTEM COMPONENT: ENVIRONMENT SETUP & REAL DATA LOADER
# =====================================================================
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🤖 Running inside Google Colab.")
    REPO_DIR = "flyrank-ml-internship-starter"
    REPO_URL = "https://github.com/Irfan-code-cloud/ML-Internship-flyrank.git"

    if not os.path.isdir(REPO_DIR):
        print(f"📥 Cloning repository from {REPO_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    if os.path.exists(REPO_DIR) and os.getcwd() != os.path.abspath(REPO_DIR):
        os.chdir(REPO_DIR)
        print(f"📂 Changed directory to repository root: {os.getcwd()}")

path_options = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv"
]

dataset_path = None
for path in path_options:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path:
    df_real = pd.read_csv(dataset_path)
    print(f"\n✅ Success! Real dataset loaded successfully from: '{dataset_path}'")
else:
    raise FileNotFoundError(f"❌ Error: Dataset still not found.")

# =====================================================================
# SYSTEM COMPONENT: SIMULATED TARGET PROXY (Empirical Expected CTR)
# =====================================================================

# Instead of a hard-coded rule, let's calculate the ACTUAL average CTR
# for each position tier directly from your dataset to serve as our expected baseline!
position_baselines = df_real.groupby('position_tier')['ctr'].mean().to_dict()

# Map the real empirical expected CTRs back to the dataframe
df_real['expected_ctr'] = df_real['position_tier'].map(position_baselines)

# Calculate the Opportunity Gap (Expected CTR - Actual Observed CTR)
df_real['ctr_gap'] = df_real['expected_ctr'] - df_real['ctr']

lane_4_columns = [
    'avg_position',
    'impressions_90d',
    'ctr',            # Observed target
    'ctr_gap',        # Calculated proxy using empirical data
    'content_type',
    'main_intent',
    'word_count'
]

df_slice = df_real[lane_4_columns].copy()

print("\n--- 📊 Real Dataset Shape ---")
print(f"🔹 Total Pages (Rows)   : {df_slice.shape[0]}")
print(f"🔹 Active Features (Cols): {df_slice.shape[1]}")

print("\n--- 🔍 Unit of Analysis: One Real Page Row ---")
single_page_row = df_slice.head(1)
pd.set_option('display.max_columns', None)
display(single_page_row)

🤖 Running inside Google Colab.
📥 Cloning repository from https://github.com/Irfan-code-cloud/ML-Internship-flyrank.git...
📂 Changed directory to repository root: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter

✅ Success! Real dataset loaded successfully from: 'data/raw/content_refresh_anonymized.csv'

--- 📊 Real Dataset Shape ---
🔹 Total Pages (Rows)   : 30000
🔹 Active Features (Cols): 7

--- 🔍 Unit of Analysis: One Real Page Row ---


,avg_position,impressions_90d,ctr,ctr_gap,content_type,main_intent,word_count
0,10.6,3803,0.76,-0.436761,keyword article,transactional,3221.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why Machine Learning Beats a Fixed Rule

A fixed rule (such as a nested `if-else` statement or a lookup table grouped solely by position) fails to solve this problem effectively for three key reasons:

#### 1. Multi-Dimensional Feature Interaction
CTR is not determined by position alone. A page ranking at position 3 targeting a **transactional** intent (where search engine results pages are crowded with paid Google Ads and shopping widgets) will naturally have a lower expected organic CTR than an **informational** page ranking at position 3.
A fixed rule cannot easily scale to calculate the interactions between:
* `avg_position` (numerical rank decay)
* `main_intent` (transactional vs. informational SERP layouts)
* `content_type` (hub page vs. keyword article)
* `word_count` (content depth/quality proxy)

#### 2. The Multi-Dimensional Search Landscape is Non-Linear
The relationship between ranking position and CTR is highly non-linear (exponential decay). Trying to write manual thresholds for every possible permutation of 44 features across different positions would result in a massive, unmaintainable codebase of hard-coded rules that would quickly break down.

#### 3. Generalization and Continuous Learning
Fixed rules are static; they do not adapt. If search intent layouts change, or if we ingest new content categories, a manual rule-based system requires an engineer to manually recalculate and rewrite the thresholds. An ML model (such as a regression tree or gradient booster) learns these multi-dimensional decision boundaries automatically from the data and can be easily retrained as new performance data flows in.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.